# Phase 2 — Launch the Slurm array

**Goal.** Turn the chunks produced by Phase 1 into a live Slurm array job, and schedule Phase 4 to run automatically once all workers finish.

**Why Python-native orchestration?** The traditional HPC approach is a static `submit.slurm` file checked into the repo. That breaks as soon as the array size depends on runtime conditions (how many chunks Phase 1 produced) or you want per-run overrides. Building the sbatch script as a Python f-string and piping it to `sbatch` via `subprocess` is simpler than templating and gives you the full language for conditional logic.

**Where to run this.** Head node, plain `python3.12` kernel. Prerequisite: Phase 1 has written at least one `data/chunks/chunk_*.csv` file.

## 1. Dry run — inspect the sbatch script *before* submitting

Pre-flight check. `--dry-run` asks the launcher to build both sbatch scripts and print them to stdout without actually submitting anything. Run this every time you change a flag — it's your chance to catch issues (wrong partition, typo in a bind path) with zero cost.

In [ ]:
!python3.12 tutorial/2_launch_slurm.py --dry-run

### What to notice in the output

**The worker array.**

- `--array=1-N%16` — N tasks total, at most 16 running at once. The `%` is Slurm's built-in throttle. Tune it with `--concurrency` when you want to be a good partition neighbor.
- `--partition=kempner_dev --account=kempner_dev` — this user's default. These two must agree (see `sacctmgr show assoc user=$USER`).
- `--gres=gpu:1` — one GPU per task; the worker is single-GPU.
- Three `--bind` mounts: `data/` for I/O, `tutorial/` for the worker script, and the AIMNet2 weights directory (read-only).
- Three `--env` flags pushing SSL cert paths through `singularity exec` so any TLS connection from inside the container (HF, PySCF basis fetches, pip) works.

**The aggregator.** The key line is `#SBATCH --dependency=afterany:<worker_jobid>`. Slurm keeps this job parked in `PENDING (Dependency)` until every array task reaches a terminal state (`COMPLETED`, `FAILED`, `CANCELLED`, ...). `afterany` is intentional: we want Phase 4 to run even if a few workers fail, so partial results are still aggregated. Switch to `afterok` if you want Phase 4 to abort on any failure.

## 2. Submit for real

Two `sbatch` calls happen in sequence: the worker array goes first, then the aggregator is submitted with the worker's job ID baked into `--dependency`. The launcher prints both job IDs so we can watch them.

In [ ]:
import subprocess, re
out = subprocess.check_output(['python3.12', 'tutorial/2_launch_slurm.py'], text=True)
print(out)
m = re.search(r'array_jobid=(\d+)\s+phase4_jobid=(\d+)', out)
array_jobid, phase4_jobid = m.group(1), m.group(2)
array_jobid, phase4_jobid

## 3. Watch the queue

Re-run the cell below every minute or so. Column meanings:

- `STATE` — `PENDING` (queued) → `RUNNING` → gone (terminal).
- `TIME` — how long the task has been in its current state.
- `NODELIST(REASON)` — assigned node or why it's still queued. Common reasons: `Priority`, `Resources`, `Dependency`, `ReqNodeNotAvail`.

The aggregator will sit in `PENDING (Dependency)` for the entire duration of the array. That's expected.

In [ ]:
!squeue -u $USER -j {array_jobid},{phase4_jobid} -o "%i %j %T %M %R"

### Cancelling a bad run

If something's clearly wrong and you want to pull the plug on both jobs at once:

```bash
scancel <array_jobid> <phase4_jobid>
```

Cancelling the array *before* the dependency resolves will also cancel Phase 4 (`DependencyNeverSatisfied`).

## 4. Confirm completion

Once both jobs leave `squeue`, use `sacct` to audit. `ExitCode=0:0` for every line = everything succeeded; anything else warrants a look at `logs/job_<jobid>_<task>.out`.

In [ ]:
!sacct -j {array_jobid},{phase4_jobid} --format=JobID,State,ExitCode,Elapsed

## 5. What landed on disk?

One `result_<N>.csv` per chunk, plus the aggregator's outputs (`stable.csv` + `phase4_summary.txt`).

In [ ]:
!ls -la data/results/ logs/

**Next:** open `04_aggregate_and_analyze.ipynb` to load the results into pandas and plot the gap distribution. If you want to understand what *happened* inside each worker, open `03_worker_demo.ipynb` (needs a GPU allocation + the container).